# Hardware sampling of the transpiled LUCJ circuits (gas)

Loads the transpiled circuits for all 10 geometry shifts from
`../../quantum_circuits/gas/charge<C>_spin<S>/transpiler_pass_initial/<N>_qubits/`,
submits them as a single `SamplerV2` job (or retrieves an existing job with `run_sampler=False` and `job_id_no`),
and writes one result file per shift to `charge<C>_spin<S>/<N>qubits/qubits_<N>_shift_<shift>_<backend>.json`
(serialised with `qiskit_ibm_runtime.RuntimeEncoder`). These files are the input of `../../sqd_post_processing/gas/`.

The last cell checks that the Hartree-Fock bitstring is present in the counts and plots the single-qubit occupation marginals.

Requires a saved IBM Quantum account (`QiskitRuntimeService.save_account(...)`). **Setting `run_sampler=True` consumes QPU time.**

In [ ]:
import os
import sys

import json
import pyscf
from pyscf import lib
from pyscf.scf import chkfile
from pyscf import ao2mo, tools 
import numpy as np
from pathlib import Path
import ffsim
import time
from qiskit import QuantumCircuit, QuantumRegister
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_aer import AerSimulator
from qiskit_addon_sqd.counts import counts_to_arrays

from matplotlib import pyplot as plt

from copy import deepcopy
import pickle
from qiskit_ibm_runtime import RuntimeEncoder,RuntimeDecoder
import qiskit.primitives.containers.bit_array as bit_array

from scipy import linalg as LA

from pyscf.mcscf import avas

from pyscf import ao2mo, tools , mcscf

import scipy

from qiskit import qpy

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

import json
from qiskit_ibm_runtime import RuntimeEncoder,RuntimeDecoder

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
)

In [ ]:
def get_circuit(num_qubits,shift,backend,DATA_DIR):
    folder=Path(DATA_DIR)
    fname=str(folder)+f'/qubits_{num_qubits}_shift_{shift}_{backend}'
    with open(f"{fname}.qpy", "rb") as file:
        qc=qpy.load(file)
    return qc[0]

### MAIN RUN

In [ ]:
service=QiskitRuntimeService()

### Parameters

| State | `charge` | `spin_number` (2S) | `num_orbitals` | `(num_elec_a, num_elec_b)` | qubits |
|---|---|---|---|---|---|
| Co(III) singlet | 3 | 0 | 17 / 19 / 23 / 25 | (11,11) / (12,12) / (14,14) / (15,15) | 34 / 38 / 46 / 50 |
| Co(III) quintet | 3 | 4 | 17 / 19 / 23 / 25 | (13,9) / (14,10) / (16,12) / (17,13) | 34 / 38 / 46 / 50 |
| Co(II) doublet | 2 | 1 | 17 | (12,11) | 34 |
| Co(II) quartet | 2 | 3 | 17 | (13,10) | 34 |

In [ ]:
shift_array=['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']
charge=3
spin_number=4
medium='gas'
num_orbitals=17
num_elec_a,num_elec_b=13,9
num_qubits=num_orbitals*2

DATA_DIR=f'../../quantum_circuits/{medium}/charge{charge}_spin{spin_number}/transpiler_pass_initial/{num_qubits}_qubits'
RESULTS_DIR=Path(f'./charge{charge}_spin{spin_number}/{num_qubits}qubits')
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
as_label = 'norb_%d_nelec_%d_%d'%(num_orbitals,num_elec_a,num_elec_b)

In [ ]:
backend_name="ibm_pittsburgh"
n_shots=200000
run_sampler=False   # True: submit a new SamplerV2 job (uses QPU time). False: retrieve an existing job.
job_id_no=None      # job ID to retrieve when run_sampler=False

In [ ]:
all_circuits=[]
for (j,shift) in enumerate(shift_array):
    print("SHIFT ",shift)
    
    isa_circuit=get_circuit(num_qubits,shift,backend_name,DATA_DIR)
    all_circuits.append(isa_circuit)

In [ ]:
job=[]
backend = service.backend(backend_name,use_fractional_gates=False)
if run_sampler==True:
    sampler = Sampler(mode=backend)
    job = sampler.run(all_circuits, shots=n_shots)
else:
    job = service.job(job_id_no)
print('job',job.job_id())

In [ ]:
job_result=job.result()

In [ ]:
for (j,shift) in enumerate(shift_array):
    print("SHIFT ",shift)
    pub_result=job_result[j]
    #if run_sampler==True:
    results_file=str(RESULTS_DIR)+f'/qubits_{num_qubits}_shift_{shift}_{backend_name}.json'
    with open(results_file,"w") as f:
        json.dump(pub_result,f,cls=RuntimeEncoder)

    print("-"*53)

In [ ]:
counts=pub_result.data.meas.get_counts()
len(counts)

In [ ]:
nq=num_qubits
for (j,shift) in enumerate(shift_array):
    print('*'*25+shift+'*'*25)
    results_file=str(RESULTS_DIR)+f'/qubits_{num_qubits}_shift_{shift}_{backend_name}.json'
    pub_result=[]
    with open(results_file) as f:
        pub_result=json.load(f,cls=RuntimeDecoder)

    counts=pub_result.data.meas.get_counts()


    hf_state=['0']*num_qubits
    for i in range(num_qubits):
        if i<num_qubits//2 and i<num_elec_a:
            hf_state[i]='1'
        if i>=num_qubits//2 and i<num_elec_b+num_qubits//2:
            hf_state[i]='1'
    final_hf_state=(''.join(np.flip(hf_state)))
    assert len(final_hf_state)==num_qubits
    print(f'State {final_hf_state} present in counts: {final_hf_state in list(counts.keys())}')

    bitwise_add = [0]*nq#50
    for bitstring in counts.keys():
        for i, ch in enumerate(bitstring):
            bitwise_add[i] += int(ch)

    plt.plot(range(nq), np.array(bitwise_add)/len(counts))
    plt.plot(np.array([int(i) for i in final_hf_state]), )
    plt.axhline(y=0.5, linestyle='--', c='black')
    plt.xlabel("Qubit index")
    plt.ylabel('Quasiprobabililty of state |1>')
    plt.grid()
    plt.show()